# LogSeer Prediction

Predicts whether a planned operation will succeed or fail based on current server logs.

**Setup**: set `DATA_DIR` to the directory containing your log sets (one subdirectory per operation run), and set the model/tokenizer paths to where your trained files are.

In [ ]:
import os, sys

# On Google Colab: clone the repo and set up the path
# On local: assumes notebook is run from the notebooks/ directory
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('logseer'):
        os.system('git clone https://github.com/masahiko-shibata/logseer.git')
    os.chdir('logseer')
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, os.path.abspath('..'))

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from logseer import Seer, print_results

In [ ]:
# Run this to load log data from Google Drive to Colab. Otherwise, copy the data folder to DATA_DIR.
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

!rm -rf logs data_current.zip 2>/dev/null
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/data/data_current.zip', 'data_current.zip')
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/models/logseer.keras', 'logseer.keras')
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/models/tokenizer.pickle', 'tokenizer.pickle')
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/models/xgb.pkl', 'xgb.pkl')
with zipfile.ZipFile('data_current.zip', 'r') as z:
    z.extractall('.')
print(os.listdir('./'))

In [ ]:
# Configuration
DATA_DIR            = '../data_current'   # directory containing log subdirectories
MODEL_PATH          = '../logseer.keras'
TOKENIZER_PATH      = '../tokenizer.pickle'
XGB_PATH            = '../xgb.pkl'
NUMCHAR             = 3000
MAX_SEQUENCE_LENGTH = 26000
NN_THRESHOLD        = 0.72
XGB_THRESHOLD       = 0.52

Run the cell below to load models and predict. Each row is one log set. `OR` triggers on either model, `AND` requires both — use AND as the high-precision restart signal.

In [ ]:
# Load models and predict
seer    = Seer.from_files(MODEL_PATH, TOKENIZER_PATH, XGB_PATH,
                          numchar=NUMCHAR, max_sequence_length=MAX_SEQUENCE_LENGTH,
                          nn_threshold=NN_THRESHOLD, xgb_threshold=XGB_THRESHOLD)
results = seer.predict(DATA_DIR)
print_results(results)